In [ ]:
# In[1]:

import pandas as pd
import numpy as np

# Load CSVs, parsing timestamp as UTC datetime (unit='s')
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_logs_df = pd.read_csv("error_logs.csv")

# Convert timestamp columns to datetime with unit='s' and UTC
metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)
trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)
log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)
error_logs_df['timestamp'] = pd.to_datetime(error_logs_df['timestamp'], unit='s', utc=True)

# Ensure value column in metric is numeric
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# Aggregate by (cmdb_id, kpi_name) using the full metric series (no time filtering)
grp = metric_df.groupby(['cmdb_id', 'kpi_name'])['value']
metric_summary = grp.agg(
    count='count',
    mean='mean',
    max='max',
    p95=lambda x: x.quantile(0.95)
).reset_index()

# Sort by count descending and keep top 20 rows; format numeric columns compactly
metric_summary['count'] = metric_summary['count'].astype(int)
metric_summary[['mean', 'max', 'p95']] = metric_summary[['mean', 'max', 'p95']].round(6)
metric_summary_top20 = metric_summary.sort_values('count', ascending=False).head(20).reset_index(drop=True)

# Distinct (cmdb_id, trace_name) pairs (top 50)
trace_pairs = trace_df[['cmdb_id', 'trace_name']].drop_duplicates().head(50).reset_index(drop=True)

# Distinct (cmdb_id, log_name) pairs (top 50)
log_pairs = log_df[['cmdb_id', 'log_name']].drop_duplicates().head(50).reset_index(drop=True)

# Display the compact summaries (keep full dataframes stored in memory)
metric_summary_top20, trace_pairs, log_pairs

```
Out[1]:
```
# Summarize findings in plain English based on the previously loaded summaries
summary = (
    "Summary of telemetry (using the full metric series):\n\n"
    "1) Data coverage\n"
    "- The top metric groups all have 25 points (uniform sampling in the returned top-20).\n"
    "- Distinct trace (cmdb_id, trace_name) pairs and log (cmdb_id, log_name) pairs were returned (top 50 each) and include many services such as ts-admin-basic-info-service, ts-admin-travel-service, ts-travel-service, and various mongo components.\n\n"
    "2) Notable metric observations (top groups by count shown):\n"
    "- ts-travel-service.diskio: extremely large I/O (mean ~7.9e9, max ~1.81e10, p95 ~1.77e10) — very high disk activity.\n"
    "- ts-travel-service.mem: high memory (mean ~4.6e8, p95 ~7.72e8, max ~7.79e8) — significant memory usage.\n"
    "- ts-travel-service.socket: elevated sockets (mean ~17.3, p95 ~31.8, max ~32.97) — many socket connections.\n"
    "- ts-travel-service latency-90: mean ~0.41s, p95 ~1.04s (max higher) — tail latency elevated for this service.\n"
    "- ts-travel2-service shows moderate CPU and disk I/O (cpu p95 ~1.84, diskio p95 ~2.27e5).\n"
    "- ts-travel2-mongo and ts-travel-mongo show notable mem and diskio values at lower magnitudes than ts-travel-service.\n"
    "- ts-voucher-service.socket is constant at 4 in the top summary (no variability).\n\n"
    "3) Next steps you may want to take\n"
    "- Investigate ts-travel-service for root-cause: high disk I/O, high memory, many sockets, and elevated latency make it the most suspicious candidate.\n"
    "- Correlate traces and logs (we loaded trace and log pairs) for ts-travel-service and related services to find errors or slow spans around incidents.\n\n"
    "If you want, I can now filter these datasets to a specific incident window and perform time-windowed anomaly detection or show the relevant trace/log pairs for ts-travel-service."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(               cmdb_id    kpi_name  count          mean           max           p95
0   ts-voucher-service      socket     25  4.000000e+00  4.000000e+00  4.000000e+00
1     ts-travel2-mongo      diskio     25  3.489930e+05  4.362701e+05  4.229324e+05
2     ts-travel2-mongo         mem     25  6.693730e+07  6.704435e+07  6.702782e+07
3     ts-travel2-mongo      socket     25  7.889333e+00  8.000000e+00  8.000000e+00
4   ts-travel2-service         cpu     25  1.346479e+00  2.687253e+00  1.844442e+00
5   ts-travel2-service      diskio     25  1.128393e+05  2.360527e+05  2.274904e+05
6   ts-travel2-service       error     25  0.000000e+00  0.000000e+00  0.000000e+00
7   ts-travel2-service  latency-50     25  4.183000e-02  4.367700e-02  4.327300e-02
8   ts-travel2-service  latency-90     25  8.007500e-02  8.506000e-02  8.489600e-02
9    ts-travel-service      diskio     25  7.907135e+09  1.811049e+10  1.766315e+10
10   ts-travel-service       error     25  0.000000e+00  0.000000e+00  0.000000e+00
11   ts-travel-service  latency-50     25  1.142310e-01  1.373185e+00  1.222240e-01
12   ts-travel-service  latency-90     25  4.081260e-01  4.245238e+00  1.037538e+00
13   ts-travel-service         mem     25  4.607978e+08  7.786364e+08  7.722180e+08
14   ts-travel-service      socket     25  1.732563e+01  3.296667e+01  3.178667e+01
15   ts-travel-service    workload     25  1.689185e+00  2.132167e+00  1.890667e+00
16    ts-travel2-mongo         cpu     25  4.900650e-01  5.234280e-01  5.223420e-01
17     ts-travel-mongo      diskio     25  3.486040e+05  4.305092e+05  4.059013e+05
18     ts-travel-mongo         mem     25  6.658229e+07  6.666993e+07  6.666603e+07
19     ts-travel-mongo      socket     25  6.988000e+00  7.000000e+00  7.000000e+00,                         cmdb_id                                         trace_name
0                          root  trace.to_ts-admin-basic-info-service.duration_...
1                          root  trace.to_ts-admin-basic-info-service.duration_p95
2                          root    trace.to_ts-admin-basic-info-service.error_rate
3                          root     trace.to_ts-admin-basic-info-service.row_count
4                          root     trace.to_ts-admin-travel-service.duration_mean
..                          ...                                                ...
45  ts-admin-basic-info-service                       trace.from_root.duration_p95
46  ts-admin-basic-info-service                         trace.from_root.error_rate
47  ts-admin-basic-info-service                          trace.from_root.row_count
48  ts-admin-basic-info-service  trace.from_ts-admin-basic-info-service.duratio...
49  ts-admin-basic-info-service  trace.from_ts-admin-basic-info-service.duratio...

[50 rows x 2 columns],                         cmdb_id         log_name
0   ts-admin-basic-info-service  log.error_count
1   ts-admin-basic-info-service    log.row_count
2       ts-admin-travel-service  log.error_count
3       ts-admin-travel-service    log.row_count
4          ts-assurance-service  log.error_count
..                          ...              ...
45            ts-food-map-mongo    log.row_count
46                ts-food-mongo  log.error_count
47                ts-food-mongo    log.row_count
48      ts-inside-payment-mongo  log.error_count
49      ts-inside-payment-mongo    log.row_count

[50 rows x 2 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

# Reuse already-loaded metric_df and metric_summary (which contains global p95 per group)

import pandas as pd

# Incident window (UTC)
start = pd.to_datetime("2024-01-25 00:25:30", utc=True)
end   = pd.to_datetime("2024-01-25 00:55:30", utc=True)

# Target cmdb_id list
targets = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]

# Windowed metrics (store as required variable)
metrics_window = metric_df[(metric_df['timestamp'] >= start) & (metric_df['timestamp'] <= end)].copy()

# Restrict to target cmdb_ids
metrics_window = metrics_window[metrics_window['cmdb_id'].isin(targets)].copy()

# Ensure numeric
metrics_window['value'] = pd.to_numeric(metrics_window['value'], errors='coerce')

# Merge global p95 thresholds computed over the full series (metric_summary has columns cmdb_id,kpi_name,p95)
p95_df = metric_summary[['cmdb_id', 'kpi_name', 'p95']].drop_duplicates()
metrics_window = metrics_window.merge(p95_df, on=['cmdb_id','kpi_name'], how='left')

# Flag anomalies using global p95 (do not recompute p95 on windowed data)
metrics_window['is_anomaly'] = metrics_window['value'] >= metrics_window['p95']

# Aggregate per (cmdb_id, kpi_name)
group_basic = metrics_window.groupby(['cmdb_id','kpi_name']).agg(
    total_points_in_window=('value','size'),
    anomaly_count=('is_anomaly','sum'),
    max_value_in_window=('value','max'),
    global_p95=('p95','first')
).reset_index()

# Earliest anomaly timestamp per group (if any)
earliest_anom = (
    metrics_window[metrics_window['is_anomaly']]
    .groupby(['cmdb_id','kpi_name'])['timestamp']
    .min()
    .reset_index()
    .rename(columns={'timestamp':'earliest_anomaly_timestamp'})
)

# Combine
anomaly_summary_window = group_basic.merge(earliest_anom, on=['cmdb_id','kpi_name'], how='left')

# Format numeric columns compactly
anomaly_summary_window['max_value_in_window'] = anomaly_summary_window['max_value_in_window'].round(6)
anomaly_summary_window['global_p95'] = anomaly_summary_window['global_p95'].round(6)

# Sort by anomaly_count desc and keep top 20
anomaly_summary_window = anomaly_summary_window.sort_values(['anomaly_count','total_points_in_window'], ascending=[False,False]).head(20).reset_index(drop=True)

# Display the requested compact summary (stored variables: metrics_window, anomaly_summary_window)
anomaly_summary_window

```
Out[2]:
```
Summary of windowed anomalies (2024-01-25 00:25:30 – 00:55:30 UTC) for the requested services:

Key findings
- ts-travel-service.error shows anomaly_count = 25 (every point). This is an artifact: global p95 for that metric is 0.0, so values >= 0 are flagged — not a meaningful signal by itself.
- Aside from that artifact, anomalies are sparse: most (cmdb_id, kpi_name) groups for ts-auth-service, ts-order-service, and ts-route-service have anomaly_count = 2 (out of 25 points in the window). That means 1–2 brief spikes exceeded the global P95 for those KPIs.
- Example earliest anomaly timestamps (UTC):
  - ts-auth-service.cpu — earliest anomaly at 2024-01-25 00:28:00 (anomaly_count 2)
  - ts-auth-service.latency-50 — earliest at 00:30:00
  - ts-auth-service.mem — earliest at 00:36:00
  - ts-order-service.cpu/diskio/latency — earliest around 00:31:00–00:32:00
  - ts-route-service latency/cpu — earliest around 00:29:00–00:30:00

Interpretation & next steps
- Most signals are short spikes (2 points) and may represent transient load or measurement noise. Focus investigation on services with repeated spikes across multiple KPIs (e.g., ts-auth-service shows spikes in cpu, mem, socket, latency).
- Treat ts-travel-service.error with caution — p95=0 makes that metric unusable for anomaly detection without adjusting threshold or logic.
- Recommended actions: inspect traces and logs around the indicated earliest anomaly timestamps for ts-auth-service, ts-order-service, and ts-route-service; verify whether the CPU/memory/diskio spikes correlate with errors or increased latency; and consider recalculating/adjusting thresholds for metrics with degenerate p95 (like the error metric).

Stored variables in the kernel: metrics_window (windowed points) and anomaly_summary_window (the compact summary shown).

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name  total_points_in_window  anomaly_count  max_value_in_window    global_p95 earliest_anomaly_timestamp
0   ts-travel-service       error                      25             25         0.000000e+00  0.000000e+00  2024-01-25 00:28:00+00:00
1     ts-auth-service         cpu                      25              2         1.650687e+01  1.551607e+01  2024-01-25 00:28:00+00:00
2     ts-auth-service      diskio                      25              2         6.436937e+04  6.253752e+04  2024-01-25 00:40:00+00:00
3     ts-auth-service  latency-50                      25              2         2.720060e-01  2.377150e-01  2024-01-25 00:30:00+00:00
4     ts-auth-service  latency-90                      25              2         8.635770e-01  8.367370e-01  2024-01-25 00:37:00+00:00
5     ts-auth-service         mem                      25              2         2.391220e+08  2.385800e+08  2024-01-25 00:36:00+00:00
6     ts-auth-service      socket                      25              2         1.476667e+01  1.464667e+01  2024-01-25 00:32:00+00:00
7     ts-auth-service    workload                      25              2         3.163250e+00  3.099217e+00  2024-01-25 00:29:00+00:00
8    ts-order-service         cpu                      25              2         9.064320e-01  7.059600e-01  2024-01-25 00:31:00+00:00
9    ts-order-service      diskio                      25              2         6.300328e+04  5.828699e+04  2024-01-25 00:31:00+00:00
10   ts-order-service  latency-50                      25              2         6.669400e-02  6.646200e-02  2024-01-25 00:32:00+00:00
11   ts-order-service  latency-90                      25              2         1.167660e-01  1.162930e-01  2024-01-25 00:32:00+00:00
12   ts-order-service         mem                      25              2         2.610202e+08  2.609433e+08  2024-01-25 00:51:00+00:00
13   ts-order-service      socket                      25              2         1.131667e+01  1.122667e+01  2024-01-25 00:36:00+00:00
14   ts-order-service    workload                      25              2         7.278000e-01  6.260300e-01  2024-01-25 00:45:00+00:00
15   ts-route-service         cpu                      25              2         1.894026e+00  1.824752e+00  2024-01-25 00:30:00+00:00
16   ts-route-service      diskio                      25              2         6.063774e+04  5.989387e+04  2024-01-25 00:37:00+00:00
17   ts-route-service  latency-50                      25              2         3.573000e-03  3.551000e-03  2024-01-25 00:29:00+00:00
18   ts-route-service  latency-90                      25              2         7.941000e-03  7.844000e-03  2024-01-25 00:29:00+00:00
19   ts-route-service         mem                      25              2         2.552276e+08  2.551879e+08  2024-01-25 00:51:00+00:00```
```

In [ ]:
# In[3]:

import pandas as pd

# Reuse existing trace_df loaded in the kernel; ensure timestamp is datetime UTC and value is numeric
trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# 1) Compute global P95 for each (cmdb_id, trace_name) across the entire series (store thresholds)
trace_p95_df = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'])['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value': 'global_p95'})
)

# 2) Filter trace.csv to the incident window and target services
start = pd.to_datetime("2024-01-25 00:25:30", utc=True)
end   = pd.to_datetime("2024-01-25 00:55:30", utc=True)
targets = ["ts-auth-service","ts-order-service","ts-route-service","ts-travel-service"]

trace_window = trace_df[(trace_df['timestamp'] >= start) & (trace_df['timestamp'] <= end) & (trace_df['cmdb_id'].isin(targets))].copy()

# 3) Merge global p95 thresholds into the windowed traces (do not recompute thresholds)
trace_window = trace_window.merge(trace_p95_df, on=['cmdb_id','trace_name'], how='left')

# Flag anomalies where value >= global_p95
trace_window['is_anomaly'] = trace_window['value'] >= trace_window['global_p95']

# 4) Aggregate stats per (cmdb_id, trace_name)
agg_basic = trace_window.groupby(['cmdb_id','trace_name']).agg(
    total_points_in_window=('value','size'),
    anomaly_count=('is_anomaly','sum'),
    max_value_in_window=('value','max'),
    mean_value_in_window=('value','mean')
).reset_index()

# Earliest anomaly timestamp per group
earliest_anom = (
    trace_window[trace_window['is_anomaly']]
    .groupby(['cmdb_id','trace_name'])['timestamp']
    .min()
    .reset_index()
    .rename(columns={'timestamp':'earliest_anomaly_timestamp'})
)

# Combine aggregates with earliest anomaly and global_p95
trace_anomaly_summary_window = (
    agg_basic
    .merge(earliest_anom, on=['cmdb_id','trace_name'], how='left')
    .merge(trace_p95_df, on=['cmdb_id','trace_name'], how='left')
)

# Format numeric columns compactly (rounding)
trace_anomaly_summary_window['max_value_in_window'] = trace_anomaly_summary_window['max_value_in_window'].round(6)
trace_anomaly_summary_window['global_p95'] = trace_anomaly_summary_window['global_p95'].round(6)
trace_anomaly_summary_window['mean_value_in_window'] = trace_anomaly_summary_window['mean_value_in_window'].round(6)

# Select and order output columns as requested
trace_anomaly_summary_window = trace_anomaly_summary_window[
    ['cmdb_id','trace_name','total_points_in_window','anomaly_count','earliest_anomaly_timestamp','max_value_in_window','global_p95']
]

# Sort by anomaly_count descending and limit to top 20 rows
trace_anomaly_summary_window = trace_anomaly_summary_window.sort_values(['anomaly_count','total_points_in_window'], ascending=[False,False]).head(20).reset_index(drop=True)

# Display the compact summary (variables trace_window and trace_anomaly_summary_window are stored in kernel)
trace_anomaly_summary_window

```
Out[3]:
```
Summary of trace-window anomalies (2024-01-25 00:25:30 – 00:55:30 UTC) for the requested services:

- What the data shows
  - All top trace entries are error_rate metrics for ts-auth-service, ts-order-service, ts-route-service, and ts-travel-service.
  - Each of these (cmdb_id, trace_name) pairs is marked as anomalous for every point in the window (anomaly_count = 25).
  - However, these flagged anomalies are an artifact: global_p95 = 0.0 and max_value_in_window = 0.0 for all those error_rate traces, so value >= p95 holds for every point (not a meaningful signal).
  - The earliest flagged timestamp for these is 2024-01-25 00:28:00 UTC (same for all listed entries).

- Interpretation
  - The error_rate traces are unusable for anomaly detection as currently thresholded (degenerate p95 = 0). They do not indicate real elevated error rates in this window — they simply meet the p95=0 threshold.
  - No meaningful anomalous trace durations or latency metrics appear in the top-20 summary returned (only error_rate series were highlighted).

- Recommended next steps
  - Exclude or re-threshold trace error_rate series with global p95 = 0 (e.g., require value > 0 or set a minimum sensible threshold) before treating them as anomalies.
  - Re-run the windowed analysis focusing on non-zero trace metrics (duration_mean, duration_p95, latency) to find real performance anomalies.
  - Correlate with logs and metrics around 2024-01-25 00:28:00 UTC if you still see other signals (metrics previously showed CPU/memory/diskio spikes for some services).

Stored kernel variables: trace_window (filtered windowed traces) and trace_anomaly_summary_window (the compact summary shown).

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                       trace_name  total_points_in_window  anomaly_count earliest_anomaly_timestamp  max_value_in_window  global_p95
0     ts-auth-service                       trace.from_root.error_rate                      25             25  2024-01-25 00:28:00+00:00                  0.0         0.0
1     ts-auth-service            trace.from_ts-auth-service.error_rate                      25             25  2024-01-25 00:28:00+00:00                  0.0         0.0
2     ts-auth-service              trace.to_ts-auth-service.error_rate                      25             25  2024-01-25 00:28:00+00:00                  0.0         0.0
3    ts-order-service                       trace.from_root.error_rate                      25             25  2024-01-25 00:28:00+00:00                  0.0         0.0
4    ts-order-service  trace.from_ts-inside-payment-service.error_rate                      25             25  2024-01-25 00:28:00+00:00                  0.0         0.0
5    ts-order-service           trace.from_ts-order-service.error_rate                      25             25  2024-01-25 00:28:00+00:00                  0.0         0.0
6    ts-order-service             trace.to_ts-order-service.error_rate                      25             25  2024-01-25 00:28:00+00:00                  0.0         0.0
7    ts-order-service           trace.to_ts-station-service.error_rate                      25             25  2024-01-25 00:28:00+00:00                  0.0         0.0
8    ts-route-service           trace.from_ts-route-service.error_rate                      25             25  2024-01-25 00:28:00+00:00                  0.0         0.0
9    ts-route-service          trace.from_ts-travel-service.error_rate                      25             25  2024-01-25 00:28:00+00:00                  0.0         0.0
10   ts-route-service         trace.from_ts-travel2-service.error_rate                      25             25  2024-01-25 00:28:00+00:00                  0.0         0.0
11   ts-route-service             trace.to_ts-route-service.error_rate                      25             25  2024-01-25 00:28:00+00:00                  0.0         0.0
12  ts-travel-service                       trace.from_root.error_rate                      25             25  2024-01-25 00:28:00+00:00                  0.0         0.0
13  ts-travel-service    trace.from_ts-admin-travel-service.error_rate                      25             25  2024-01-25 00:28:00+00:00                  0.0         0.0
14  ts-travel-service            trace.from_ts-food-service.error_rate                      25             25  2024-01-25 00:28:00+00:00                  0.0         0.0
15  ts-travel-service          trace.from_ts-travel-service.error_rate                      25             25  2024-01-25 00:28:00+00:00                  0.0         0.0
16  ts-travel-service             trace.to_ts-route-service.error_rate                      25             25  2024-01-25 00:28:00+00:00                  0.0         0.0
17  ts-travel-service        trace.to_ts-ticketinfo-service.error_rate                      25             25  2024-01-25 00:28:00+00:00                  0.0         0.0
18  ts-travel-service             trace.to_ts-train-service.error_rate                      25             25  2024-01-25 00:28:00+00:00                  0.0         0.0
19  ts-travel-service            trace.to_ts-travel-service.error_rate                      25             25  2024-01-25 00:28:00+00:00                  0.0         0.0```
```

In [ ]:
# In[4]:

import pandas as pd

# Ensure timestamps and numeric values are correctly typed (reuse existing variables if present)
log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)
log_df['value'] = pd.to_numeric(log_df['value'], errors='coerce')

error_logs_df['timestamp'] = pd.to_datetime(error_logs_df['timestamp'], unit='s', utc=True)

# Compute global P95 for each (cmdb_id, log_name) across the full series (before any window filtering)
log_p95_df = (
    log_df
    .groupby(['cmdb_id', 'log_name'])['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value': 'global_p95'})
)

# Incident window (UTC) and target services
start = pd.to_datetime("2024-01-25 00:25:30", utc=True)
end   = pd.to_datetime("2024-01-25 00:55:30", utc=True)
targets = ["ts-auth-service","ts-order-service","ts-route-service"]

# Filter logs to the window and target services
logs_window = log_df[(log_df['timestamp'] >= start) & (log_df['timestamp'] <= end) & (log_df['cmdb_id'].isin(targets))].copy()

# Merge global p95 thresholds into the windowed logs
logs_window = logs_window.merge(log_p95_df, on=['cmdb_id','log_name'], how='left')

# Flag anomalies where value >= global_p95
logs_window['is_anomaly'] = logs_window['value'] >= logs_window['global_p95']

# Aggregate per (cmdb_id, log_name)
agg_basic = logs_window.groupby(['cmdb_id','log_name']).agg(
    total_points_in_window=('value','size'),
    anomaly_count=('is_anomaly','sum'),
    max_value_in_window=('value','max')
).reset_index()

# Earliest anomaly timestamp per group (if any)
earliest_anom = (
    logs_window[logs_window['is_anomaly']]
    .groupby(['cmdb_id','log_name'])['timestamp']
    .min()
    .reset_index()
    .rename(columns={'timestamp':'earliest_anomaly_timestamp'})
)

# Combine aggregates with earliest anomaly and global_p95
log_anomaly_summary_window = (
    agg_basic
    .merge(earliest_anom, on=['cmdb_id','log_name'], how='left')
    .merge(log_p95_df, on=['cmdb_id','log_name'], how='left')
)

# Round numeric columns for compact display
log_anomaly_summary_window['max_value_in_window'] = log_anomaly_summary_window['max_value_in_window'].round(6)
log_anomaly_summary_window['global_p95'] = log_anomaly_summary_window['global_p95'].round(6)

# Select and order output columns
log_anomaly_summary_window = log_anomaly_summary_window[
    ['cmdb_id','log_name','total_points_in_window','anomaly_count','earliest_anomaly_timestamp','max_value_in_window','global_p95']
]

# Sort by anomaly_count descending and limit to top 20 rows
log_anomaly_summary_window = log_anomaly_summary_window.sort_values(['anomaly_count','total_points_in_window'], ascending=[False,False]).head(20).reset_index(drop=True)

# Filter error_logs to same window and target services, select requested columns, sort and keep up to first 20 rows
error_logs_window = (
    error_logs_df[(error_logs_df['timestamp'] >= start) & (error_logs_df['timestamp'] <= end) & (error_logs_df['cmdb_id'].isin(targets))]
    .loc[:, ['timestamp','cmdb_id','message']]
    .sort_values('timestamp')
    .head(20)
    .reset_index(drop=True)
)

# Display the compact summaries (variables stored in kernel: logs_window, log_anomaly_summary_window, error_logs_window)
log_anomaly_summary_window, error_logs_window

```
Out[4]:
```
Summary of log analysis for 2024-01-25 00:25:30–00:55:30 UTC (ts-auth, ts-order, ts-route):

Key findings
- log.error_count:
  - All three services (ts-auth-service, ts-order-service, ts-route-service) show anomaly_count = 25 because global_p95 = 0.0 and all values are >= 0. This is a degenerate threshold and not a meaningful signal (max_value_in_window = 0 for these series). Earliest flagged time: 2024-01-25 00:28:00 UTC. Treat these as artifacts unless thresholds are adjusted.

- log.row_count:
  - ts-order-service: total_points = 25, anomaly_count = 3, earliest anomaly at 2024-01-25 00:36:00 UTC, max_value_in_window = 62, global_p95 = 55.0.
  - ts-auth-service: total_points = 25, anomaly_count = 2, earliest anomaly at 2024-01-25 00:29:00 UTC, max_value_in_window = 380, global_p95 = 374.4.
  - ts-route-service: total_points = 25, anomaly_count = 2, earliest anomaly at 2024-01-25 00:29:00 UTC, max_value_in_window = 382, global_p95 = 372.8.
  - Interpretation: these are brief spikes above the historical 95th percentile for log volume/row counts and merit inspection (could indicate bursts of traffic, logging surges, or upstream events).

- error_logs:
  - No raw error log entries were found in the incident window for the three candidate services (error_logs_window is empty).

Recommendations
- Ignore or re-threshold log.error_count (p95 = 0) before treating it as an anomaly.
- Investigate the log.row_count spikes around 00:29:00–00:36:00 UTC for ts-auth, ts-route, and ts-order: check corresponding traces and metrics for correlated CPU/memory/disk or latency spikes and review application logs around those timestamps.
- Stored kernel variables you can use next: logs_window (windowed log points), log_anomaly_summary_window (compact summary shown), error_logs_window (filtered raw error logs — empty).

The original code execution output of IPython Kernel is also provided below for reference:

(            cmdb_id         log_name  total_points_in_window  anomaly_count earliest_anomaly_timestamp  max_value_in_window  global_p95
0   ts-auth-service  log.error_count                      25             25  2024-01-25 00:28:00+00:00                    0         0.0
1  ts-order-service  log.error_count                      25             25  2024-01-25 00:28:00+00:00                    0         0.0
2  ts-route-service  log.error_count                      25             25  2024-01-25 00:28:00+00:00                    0         0.0
3  ts-order-service    log.row_count                      25              3  2024-01-25 00:36:00+00:00                   62        55.0
4   ts-auth-service    log.row_count                      25              2  2024-01-25 00:29:00+00:00                  380       374.4
5  ts-route-service    log.row_count                      25              2  2024-01-25 00:29:00+00:00                  382       372.8, Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[5]:

import pandas as pd

# Reuse trace_df already loaded in the kernel; ensure types
trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# Incident window and target services
start = pd.to_datetime("2024-01-25 00:25:30", utc=True)
end   = pd.to_datetime("2024-01-25 00:55:30", utc=True)
targets = ["ts-auth-service","ts-order-service","ts-route-service","ts-travel-service"]

# Windowed traces for target services (store as trace_window to reuse)
trace_window = trace_df[(trace_df['timestamp'] >= start) & (trace_df['timestamp'] <= end) & (trace_df['cmdb_id'].isin(targets))].copy()

# Substring filters of interest
substrs = [
    'to_ts-auth-service','from_ts-auth-service',
    'to_ts-order-service','from_ts-order-service',
    'to_ts-route-service','from_ts-route-service',
    'to_ts-travel-service','from_ts-travel-service',
    'duration','duration_mean','duration_p95','latency'
]
pattern = "(" + "|".join(substrs) + ")"

# Filter traces whose trace_name contains any of the substrings
candidate_trace_window = trace_window[trace_window['trace_name'].str.contains(pattern, na=False)].copy()

# Aggregate per distinct trace_name
candidate_trace_summary = (
    candidate_trace_window
    .groupby('trace_name', as_index=False)
    .agg(
        total_points_in_window=('value','size'),
        mean_value_in_window=('value','mean'),
        max_value_in_window=('value','max'),
        earliest_timestamp_in_window=('timestamp','min')
    )
)

# Format numeric columns compactly and sort by total_points descending, limit to 50 rows
candidate_trace_summary['mean_value_in_window'] = candidate_trace_summary['mean_value_in_window'].round(6)
candidate_trace_summary['max_value_in_window'] = candidate_trace_summary['max_value_in_window'].round(6)
candidate_trace_summary = candidate_trace_summary.sort_values('total_points_in_window', ascending=False).head(50).reset_index(drop=True)

# Also produce a small list (up to 20) of unique trace_name values that mention calls between candidate services
# (i.e., trace_name contains any of the candidate service names)
service_pattern = "(" + "|".join(targets) + ")"
interservice_traces = (
    candidate_trace_window[candidate_trace_window['trace_name'].str.contains(service_pattern, na=False)]
    ['trace_name']
    .drop_duplicates()
    .head(20)
    .reset_index(drop=True)
    .to_frame(name='trace_name')
)

# Store variables in kernel as requested:
# - candidate_trace_window (filtered rows)
# - candidate_trace_summary (aggregated summary)
# Also keep trace_window available (already defined)

# Display compact summaries: candidate_trace_summary (max 50 rows) and interservice_traces (max 20 rows)
candidate_trace_summary, interservice_traces

```
Out[5]:
```
Summary of candidate traces in the incident window (2024-01-25 00:25:30–00:55:30 UTC):

What we found
- Distinct candidate trace names (up to 50) were extracted for the four services. Many trace types are present: duration, duration_mean, duration_p95, latency-related, row_count, and error_rate for calls both to- and from- candidate services.
- Top aggregated items (examples from the compact table):
  - trace.from_root.duration_mean — 75 points, mean ~0.161s, max ~0.415s, earliest 2024-01-25 00:28:00 UTC.
  - trace.from_root.duration_p95 — 75 points, mean ~0.340s, max ~1.003s, earliest 00:28:00.
  - trace.to_ts-route-service.row_count — 50 points, mean ~385 rows, max 764, earliest 00:28:00.
  - Many inter-service duration traces (e.g., trace.from_ts-travel-service.duration_mean, trace.to_ts-travel-service.duration_p95, trace.to_ts-train-service.duration_p95) appear with 25–50 points and small mean durations (ms–tens of ms), earliest at 00:28:00.
- Inter-service trace names (up to 20) explicitly mentioning calls among candidate services were listed (examples include):
  - trace.from_ts-auth-service.duration_mean / duration_p95 / row_count / error_rate
  - trace.to_ts-auth-service.duration_mean / duration_p95 / row_count / error_rate
  - trace.from_ts-order-service.* and trace.to_ts-order-service.*
  - trace.from_ts-route-service.* and trace.to_ts-route-service.*

Interpretation and next steps
- A common earliest timestamp is 2024-01-25 00:28:00 UTC for many traces — this aligns with earlier metric/log anomalies and is the right time to investigate.
- Duration metrics are generally small (ms scale) but trace.from_root.duration_p95 shows a higher tail (max ~1.0s) — worth inspecting root/span traces for slow tails.
- Row_count traces (e.g., to_ts-route-service.row_count) show large counts and spikes (maxs several hundred), matching the log.row_count anomalies previously observed — suggests traffic bursts or heavy load toward route service.
- Recommended actions:
  - Inspect full spans for trace.from_root.duration_p95 at/after 00:28:00 to find slow paths and which downstream calls contribute.
  - Correlate inter-service traces (auth↔order↔route↔travel) and the metric/log spikes around 00:28–00:36 to identify which call flows coincide with latency or load spikes.
  - Use candidate_trace_window and candidate_trace_summary (stored in the kernel) to drill into individual spans and timestamps.

Stored kernel variables you can reuse:
- candidate_trace_window (filtered trace rows for the window and substrings)
- candidate_trace_summary (aggregated compact summary shown)

The original code execution output of IPython Kernel is also provided below for reference:

(                                    trace_name  total_points_in_window  mean_value_in_window  max_value_in_window earliest_timestamp_in_window
0                trace.from_root.duration_mean                      75              0.161266             0.414645    2024-01-25 00:28:00+00:00
1                 trace.from_root.duration_p95                      75              0.339708             1.003437    2024-01-25 00:28:00+00:00
2          trace.to_ts-route-service.row_count                      50            384.980000           764.000000    2024-01-25 00:28:00+00:00
3         trace.to_ts-route-service.error_rate                      50              0.000000             0.000000    2024-01-25 00:28:00+00:00
4   trace.from_ts-travel-service.duration_mean                      50              0.005849             0.013273    2024-01-25 00:28:00+00:00
..                                         ...                     ...                   ...                  ...                          ...
43      trace.to_ts-train-service.duration_p95                      25              0.005309             0.009944    2024-01-25 00:28:00+00:00
44    trace.to_ts-travel-service.duration_mean                      25              0.008497             0.013273    2024-01-25 00:28:00+00:00
45     trace.to_ts-travel-service.duration_p95                      25              0.038804             0.080396    2024-01-25 00:28:00+00:00
46       trace.to_ts-travel-service.error_rate                      25              0.000000             0.000000    2024-01-25 00:28:00+00:00
47        trace.to_ts-travel-service.row_count                      25            314.200000           411.000000    2024-01-25 00:28:00+00:00

[48 rows x 5 columns],                                    trace_name
0    trace.from_ts-auth-service.duration_mean
1     trace.from_ts-auth-service.duration_p95
2       trace.from_ts-auth-service.error_rate
3        trace.from_ts-auth-service.row_count
4      trace.to_ts-auth-service.duration_mean
5       trace.to_ts-auth-service.duration_p95
6         trace.to_ts-auth-service.error_rate
7          trace.to_ts-auth-service.row_count
8   trace.from_ts-order-service.duration_mean
9    trace.from_ts-order-service.duration_p95
10     trace.from_ts-order-service.error_rate
11      trace.from_ts-order-service.row_count
12    trace.to_ts-order-service.duration_mean
13     trace.to_ts-order-service.duration_p95
14       trace.to_ts-order-service.error_rate
15        trace.to_ts-order-service.row_count
16  trace.from_ts-route-service.duration_mean
17   trace.from_ts-route-service.duration_p95
18     trace.from_ts-route-service.error_rate
19      trace.from_ts-route-service.row_count)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

import pandas as pd

# Reuse existing trace_df if present; ensure timestamp and value types
trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# Target services and substrings of interest
targets = ["ts-auth-service","ts-order-service","ts-route-service","ts-travel-service"]
substrs = ['duration','duration_mean','duration_p95','latency']
pattern = "(" + "|".join(substrs) + ")"

# Identify matching (cmdb_id, trace_name) pairs in the full series for the target services
matches = trace_df[
    trace_df['cmdb_id'].isin(targets) &
    trace_df['trace_name'].str.contains(pattern, na=False)
][['cmdb_id','trace_name']].drop_duplicates()

# Compute global P95 over the entire trace_df for these pairs (per rule: thresholds computed before window filtering)
trace_p95_full = (
    trace_df
    .merge(matches, on=['cmdb_id','trace_name'], how='inner')  # limit grouping to matching pairs
    .groupby(['cmdb_id','trace_name'])['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value':'global_p95'})
)

# Incident window (UTC)
start = pd.to_datetime("2024-01-25 00:25:30", utc=True)
end   = pd.to_datetime("2024-01-25 00:55:30", utc=True)

# Filter trace rows to the incident window and target services and matching trace_names
trace_duration_window = trace_df[
    (trace_df['timestamp'] >= start) &
    (trace_df['timestamp'] <= end) &
    (trace_df['cmdb_id'].isin(targets)) &
    (trace_df['trace_name'].str.contains(pattern, na=False))
].copy()

# Merge global p95 thresholds (computed from full series) into the windowed traces
trace_duration_window = trace_duration_window.merge(trace_p95_full, on=['cmdb_id','trace_name'], how='left')

# Flag anomalies where value >= global_p95 (handle NaN global_p95 safely)
trace_duration_window['is_anomaly'] = False
mask_has_p95 = trace_duration_window['global_p95'].notna()
trace_duration_window.loc[mask_has_p95, 'is_anomaly'] = trace_duration_window.loc[mask_has_p95, 'value'] >= trace_duration_window.loc[mask_has_p95, 'global_p95']

# Aggregate required stats per (cmdb_id, trace_name)
agg_basic = trace_duration_window.groupby(['cmdb_id','trace_name']).agg(
    total_points_in_window=('value','size'),
    anomaly_count=('is_anomaly','sum'),
    max_value_in_window=('value','max'),
    mean_value_in_window=('value','mean')
).reset_index()

# Earliest anomaly timestamp per group (NaT if none)
earliest_anom = (
    trace_duration_window[trace_duration_window['is_anomaly']]
    .groupby(['cmdb_id','trace_name'])['timestamp']
    .min()
    .reset_index()
    .rename(columns={'timestamp':'earliest_anomaly_timestamp'})
)

# Combine aggregates with earliest anomaly and global_p95
trace_duration_anomaly_summary = (
    agg_basic
    .merge(earliest_anom, on=['cmdb_id','trace_name'], how='left')
    .merge(trace_p95_full, on=['cmdb_id','trace_name'], how='left')
)

# Keep requested output columns and formatting; drop mean_value_in_window (not requested in final output)
trace_duration_anomaly_summary = trace_duration_anomaly_summary[
    ['cmdb_id','trace_name','total_points_in_window','anomaly_count','earliest_anomaly_timestamp','max_value_in_window','global_p95']
]

# Round numeric columns for compactness
trace_duration_anomaly_summary['max_value_in_window'] = trace_duration_anomaly_summary['max_value_in_window'].round(6)
trace_duration_anomaly_summary['global_p95'] = trace_duration_anomaly_summary['global_p95'].round(6)

# Sort by anomaly_count descending and limit to top 20 rows
trace_duration_anomaly_summary = trace_duration_anomaly_summary.sort_values(['anomaly_count','total_points_in_window'], ascending=[False,False]).head(20).reset_index(drop=True)

# Display the compact summary (store full windowed traces in trace_duration_window)
trace_duration_anomaly_summary

```
Out[6]:
```
Summary of duration/latency trace anomalies (using full-series P95 thresholds; window 2024-01-25 00:25:30–00:55:30 UTC):

What was checked
- We examined trace names containing 'duration', 'duration_mean', 'duration_p95', or 'latency' for the four services: ts-auth-service, ts-order-service, ts-route-service, ts-travel-service.
- Global P95 thresholds were computed from the full trace series, then the window was scanned for values >= those P95 thresholds.

Key findings (compact)
- Most matching trace series show only a few spikes in the window (anomaly_count = 2 for the top rows), i.e., short-lived excursions above historical P95.
- Notable high-tail trace:
  - ts-auth-service: trace.from_root.duration_p95 — max in window ~1.003 s, global P95 ~0.9889, earliest anomalous point 2024-01-25 00:28:00 UTC. This indicates a meaningful tail-latency spike for root spans.
  - ts-auth-service: trace.from_root.duration_mean — max ~0.4146 s, global P95 ~0.3659, earliest anomaly 00:28:00.
- Other duration spikes are much smaller (milliseconds scale) but exceed their historical P95s:
  - ts-auth-service inter-service durations (from/to ts-auth-service) had anomalies with earliest ~2024-01-25 00:48:00.
  - ts-order-service from_root.duration_* anomalies earliest ~00:31:00; various to/from order-service and to station-service duration_p95 show earliest anomalies at ~00:28–00:31.
  - ts-route-service durations show small absolute values (ms) but had two anomalous points, earliest ~00:28–00:29.
- All listed entries have total_points_in_window = 25 and anomaly_count = 2 (except where indicated), meaning anomalies are brief (2 samples out of 25).

Interpretation and recommended next steps
- The most actionable signal is the ts-auth-service root-span tail (trace.from_root.duration_p95) with ~1s max and P95 near 0.99s at 00:28:00 — investigate full spans and which downstream calls contributed to that tail.
- Correlate the noted earliest anomaly timestamps (notably ~00:28:00 and ~00:31:00 and ~00:48:00) with metrics and logs for the same services to determine cause (CPU/memory spikes, log.row_count bursts, or inter-service call congestion).
- Drill into trace_duration_window and trace_duration_anomaly_summary (stored in the kernel) to inspect full span records around those earliest anomalous timestamps.

Stored kernel variables you can reuse:
- trace_duration_window (windowed duration/latency trace rows)
- trace_duration_anomaly_summary (compact summary shown)

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                         trace_name  total_points_in_window  anomaly_count earliest_anomaly_timestamp  max_value_in_window  global_p95
0    ts-auth-service                      trace.from_root.duration_mean                      25              2  2024-01-25 00:28:00+00:00             0.414645    0.365941
1    ts-auth-service                       trace.from_root.duration_p95                      25              2  2024-01-25 00:28:00+00:00             1.003437    0.988912
2    ts-auth-service           trace.from_ts-auth-service.duration_mean                      25              2  2024-01-25 00:48:00+00:00             0.005301    0.004226
3    ts-auth-service            trace.from_ts-auth-service.duration_p95                      25              2  2024-01-25 00:48:00+00:00             0.023636    0.004936
4    ts-auth-service             trace.to_ts-auth-service.duration_mean                      25              2  2024-01-25 00:48:00+00:00             0.005301    0.004226
5    ts-auth-service              trace.to_ts-auth-service.duration_p95                      25              2  2024-01-25 00:48:00+00:00             0.023636    0.004936
6   ts-order-service                      trace.from_root.duration_mean                      25              2  2024-01-25 00:31:00+00:00             0.091420    0.085193
7   ts-order-service                       trace.from_root.duration_p95                      25              2  2024-01-25 00:31:00+00:00             0.128857    0.117709
8   ts-order-service  trace.from_ts-inside-payment-service.duration_...                      25              2  2024-01-25 00:35:00+00:00             0.010113    0.007759
9   ts-order-service  trace.from_ts-inside-payment-service.duration_p95                      25              2  2024-01-25 00:35:00+00:00             0.029601    0.012017
10  ts-order-service          trace.from_ts-order-service.duration_mean                      25              2  2024-01-25 00:28:00+00:00             0.021257    0.020713
11  ts-order-service           trace.from_ts-order-service.duration_p95                      25              2  2024-01-25 00:28:00+00:00             0.080255    0.079341
12  ts-order-service            trace.to_ts-order-service.duration_mean                      25              2  2024-01-25 00:28:00+00:00             0.021257    0.020713
13  ts-order-service             trace.to_ts-order-service.duration_p95                      25              2  2024-01-25 00:28:00+00:00             0.080255    0.079341
14  ts-order-service          trace.to_ts-station-service.duration_mean                      25              2  2024-01-25 00:28:00+00:00             0.074914    0.070397
15  ts-order-service           trace.to_ts-station-service.duration_p95                      25              2  2024-01-25 00:31:00+00:00             0.094158    0.082357
16  ts-route-service          trace.from_ts-route-service.duration_mean                      25              2  2024-01-25 00:28:00+00:00             0.001331    0.001262
17  ts-route-service           trace.from_ts-route-service.duration_p95                      25              2  2024-01-25 00:29:00+00:00             0.001760    0.001691
18  ts-route-service         trace.from_ts-travel-service.duration_mean                      25              2  2024-01-25 00:28:00+00:00             0.003526    0.003345
19  ts-route-service          trace.from_ts-travel-service.duration_p95                      25              2  2024-01-25 00:28:00+00:00             0.004948    0.003993```
```